In [1]:
import pandas as pd

train = pd.read_excel("case_1_labeled_data.xlsx")

train.head()

,full_text,label
0,@ARSIPAJA Pret. Di sekolah gw dapet MBG tetep ...,Sasaran Penerima
1,MBG bentuk penggarongan duit negara secara TSM...,Politik
2,@inzhapt_76 @ARSIPAJA Pasal 34 ayat (1) Undang...,Sasaran Penerima
3,Makan Bergizi Gratis bikin masyarakat ngerasa ...,Sasaran Penerima
4,"@OniSuryaman Presiden ngotot, paling sebel kal...",Politik


In [2]:
train['label'].value_counts()

label
Kualitas Pangan     1247
Politik              792
Anggaran             727
Lainnya              638
Tata Kelola          511
Sasaran Penerima     507
Distribusi           433
Ekonomi              145
Name: count, dtype: int64

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    train['full_text'],
    train['label'],
    test_size=0.2,
    stratify=train['label'],
    random_state=42
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=1
    )),
    ('clf', LinearSVC(
        class_weight='balanced',
        C=1.5
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_val)

score = balanced_accuracy_score(y_val, pred)

print("Balanced Accuracy:", score)

Balanced Accuracy: 0.6028651114435702


In [12]:
import re

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'http\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)

    text = re.sub(r'#(\w+)', r'\1', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

train['clean_text'] = train['clean_text'].apply(clean_text)
X_train, X_val, y_train, y_val = train_test_split(
    train['clean_text'],
    train['label'],
    test_size=0.2,
    stratify=train['label'],
    random_state=42
)

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    train['clean_text'],
    train['label'],
    test_size=0.2,
    stratify=train['label'],
    random_state=42
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=1
    )),
    ('clf', LinearSVC(
        class_weight='balanced',
        C=1.5
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_val)

score = balanced_accuracy_score(y_val, pred)

print("Balanced Accuracy:", score)

Balanced Accuracy: 0.592098707434773


In [11]:
train[['full_text', 'clean_text']].head()

,full_text,clean_text
0,@ARSIPAJA Pret. Di sekolah gw dapet MBG tetep ...,user pret. di sekolah gw dapet mbg tetep aja a...
1,MBG bentuk penggarongan duit negara secara TSM...,mbg bentuk penggarongan duit negara secara tsm...
2,@inzhapt_76 @ARSIPAJA Pasal 34 ayat (1) Undang...,user user pasal 34 ayat (1) undang-undang dasa...
3,Makan Bergizi Gratis bikin masyarakat ngerasa ...,makan bergizi gratis bikin masyarakat ngerasa ...
4,"@OniSuryaman Presiden ngotot, paling sebel kal...","user presiden ngotot, paling sebel kalau mbg d..."


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_val, pred))

                  precision    recall  f1-score   support

        Anggaran       0.67      0.77      0.72       145
      Distribusi       0.68      0.63      0.65        87
         Ekonomi       0.63      0.66      0.64        29
 Kualitas Pangan       0.65      0.74      0.70       249
         Lainnya       0.50      0.44      0.47       128
         Politik       0.56      0.47      0.51       159
Sasaran Penerima       0.55      0.53      0.54       101
     Tata Kelola       0.54      0.49      0.51       102

        accuracy                           0.61      1000
       macro avg       0.60      0.59      0.59      1000
    weighted avg       0.60      0.61      0.60      1000



In [25]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

# WORD TFIDF
word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1,2),
    min_df=1,
    lowercase=True
)

# CHARACTER TFIDF
char_tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    min_df=1,
    lowercase=True
)

# GABUNG FEATURE
features = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

# MODEL
model = Pipeline([
    ('features', features),
    ('clf', LinearSVC(
        class_weight='balanced',
        C=1.5
    ))
])

# CROSS VALIDATION
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    train['full_text'],
    train['label'],
    cv=cv,
    scoring='balanced_accuracy'
)

print("CV Scores:", scores)
print("Mean CV Score:", scores.mean())

CV Scores: [0.57606007 0.57728625 0.62094867 0.6214492  0.61252382]
Mean CV Score: 0.6016536037314704
